In [2]:
!pip install -q pandas numpy scipy scikit-learn joblib

In [3]:
from google.colab import files

uploaded = files.upload()

Saving links.csv to links.csv
Saving movies.csv to movies.csv
Saving ratings.csv to ratings.csv
Saving tags.csv to tags.csv


In [4]:
import os

print(os.listdir())

['.config', 'links.csv', 'ratings.csv', 'tags.csv', 'movies.csv', 'sample_data']


In [5]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [6]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)

display(movies.head())
display(ratings.head())
display(tags.head())

Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [7]:
movies = movies.drop_duplicates("movieId").copy()

ratings = ratings.dropna(
    subset=["userId", "movieId", "rating"]
).copy()

tags = tags.dropna(
    subset=["movieId", "tag"]
).copy()

ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)

tags["movieId"] = tags["movieId"].astype(int)

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)

Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)


In [8]:
tag_features = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

tag_features.rename(
    columns={"tag": "tag_text"},
    inplace=True
)

tag_features.head()

,movieId,tag_text
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


In [9]:
content_df = movies.merge(
    tag_features,
    on="movieId",
    how="left"
)

content_df["tag_text"] = content_df["tag_text"].fillna("")
content_df["genres"] = content_df["genres"].fillna("")

content_df["genres_text"] = (
    content_df["genres"]
    .str.replace("|", " ", regex=False)
)

content_df["content_text"] = (
    content_df["genres_text"] + " " +
    content_df["tag_text"]
)

content_df[[
    "movieId",
    "title",
    "genres",
    "content_text"
]].head()

,movieId,title,genres,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy pi...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy fantasy magic board...
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy pregnancy remake


In [10]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf.fit_transform(
    content_df["content_text"]
)

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (9742, 5528)


In [11]:
from sklearn.preprocessing import normalize

# Convert TF-IDF to dense only if dataset size allows it
tfidf_dense = tfidf_matrix.toarray().astype(np.float32)

print(tfidf_dense.shape)

(9742, 5528)


In [12]:
tfidf_centered = (
    tfidf_dense -
    tfidf_dense.mean(axis=1, keepdims=True)
)

norms = np.linalg.norm(
    tfidf_centered,
    axis=1,
    keepdims=True
)

norms[norms == 0] = 1

tfidf_normalized = (
    tfidf_centered / norms
)

print("Normalized CBF matrix:", tfidf_normalized.shape)

Normalized CBF matrix: (9742, 5528)


In [13]:
TOP_N = 50

cbf_indices = []
cbf_scores = []

In [14]:
for start in range(0, len(content_df), 500):

    end = min(
        start + 500,
        len(content_df)
    )

    batch = tfidf_normalized[start:end]

    similarity = batch @ tfidf_normalized.T

    for i in range(similarity.shape[0]):

        row = similarity[i]

        # Ignore itself
        global_index = start + i
        row[global_index] = -1

        top_indices = np.argpartition(
            row,
            -TOP_N
        )[-TOP_N:]

        top_indices = top_indices[
            np.argsort(row[top_indices])[::-1]
        ]

        cbf_indices.append(top_indices)
        cbf_scores.append(
            row[top_indices]
        )

    print(f"Processed {end}/{len(content_df)} movies")

Processed 500/9742 movies
Processed 1000/9742 movies
Processed 1500/9742 movies
Processed 2000/9742 movies
Processed 2500/9742 movies
Processed 3000/9742 movies
Processed 3500/9742 movies
Processed 4000/9742 movies
Processed 4500/9742 movies
Processed 5000/9742 movies
Processed 5500/9742 movies
Processed 6000/9742 movies
Processed 6500/9742 movies
Processed 7000/9742 movies
Processed 7500/9742 movies
Processed 8000/9742 movies
Processed 8500/9742 movies
Processed 9000/9742 movies
Processed 9500/9742 movies
Processed 9742/9742 movies


In [15]:
cbf_indices = np.array(cbf_indices)
cbf_scores = np.array(cbf_scores)

print(cbf_indices.shape)
print(cbf_scores.shape)

(9742, 50)
(9742, 50)


In [16]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print(user_movie_matrix.shape)

(610, 9724)


In [17]:
user_means = user_movie_matrix.mean(
    axis=1
)

rating_centered = (
    user_movie_matrix
    .sub(user_means, axis=0)
)

rating_centered = rating_centered.fillna(0)

print(rating_centered.shape)

(610, 9724)


In [18]:
rating_matrix = rating_centered.values.astype(
    np.float32
)

print(rating_matrix.shape)

(610, 9724)


In [19]:
movie_vectors = rating_matrix.T

movie_norms = np.linalg.norm(
    movie_vectors,
    axis=1,
    keepdims=True
)

movie_norms[movie_norms == 0] = 1

movie_normalized = (
    movie_vectors / movie_norms
)

In [20]:
cf_indices = []
cf_scores = []

movie_count = movie_normalized.shape[0]

In [21]:
for start in range(0, movie_count, 500):

    end = min(
        start + 500,
        movie_count
    )

    batch = movie_normalized[start:end]

    similarity = batch @ movie_normalized.T

    for i in range(similarity.shape[0]):

        global_index = start + i

        row = similarity[i]

        row[global_index] = -1

        top_indices = np.argpartition(
            row,
            -TOP_N
        )[-TOP_N:]

        top_indices = top_indices[
            np.argsort(row[top_indices])[::-1]
        ]

        cf_indices.append(top_indices)
        cf_scores.append(
            row[top_indices]
        )

    print(f"Processed {end}/{movie_count} movies")

Processed 500/9724 movies
Processed 1000/9724 movies
Processed 1500/9724 movies
Processed 2000/9724 movies
Processed 2500/9724 movies
Processed 3000/9724 movies
Processed 3500/9724 movies
Processed 4000/9724 movies
Processed 4500/9724 movies
Processed 5000/9724 movies
Processed 5500/9724 movies
Processed 6000/9724 movies
Processed 6500/9724 movies
Processed 7000/9724 movies
Processed 7500/9724 movies
Processed 8000/9724 movies
Processed 8500/9724 movies
Processed 9000/9724 movies
Processed 9500/9724 movies
Processed 9724/9724 movies


In [22]:
cf_indices = np.array(cf_indices)
cf_scores = np.array(cf_scores)

print("CF indices:", cf_indices.shape)
print("CF scores:", cf_scores.shape)

CF indices: (9724, 50)
CF scores: (9724, 50)


In [23]:
movie_ids = content_df["movieId"].values

movie_id_to_index = {
    int(movie_id): int(index)
    for index, movie_id
    in enumerate(movie_ids)
}

index_to_movie_id = {
    int(index): int(movie_id)
    for index, movie_id
    in enumerate(movie_ids)
}

In [24]:
joblib.dump(
    movie_id_to_index,
    "movie_id_to_index.pkl"
)

joblib.dump(
    index_to_movie_id,
    "index_to_movie_id.pkl"
)

['index_to_movie_id.pkl']

In [25]:
rating_counts = (
    ratings
    .groupby("movieId")
    .size()
)

rating_counts = rating_counts.to_dict()

joblib.dump(
    rating_counts,
    "rating_counts.pkl"
)

['rating_counts.pkl']

In [26]:
user_rating_counts = (
    ratings
    .groupby("userId")
    .size()
)

user_rating_counts = (
    user_rating_counts
    .to_dict()
)

joblib.dump(
    user_rating_counts,
    "user_rating_counts.pkl"
)

['user_rating_counts.pkl']

In [27]:
ALPHA = 0.5

In [28]:
def get_cbf_candidates(movie_id, top_n=50):

    if movie_id not in movie_id_to_index:
        return []

    idx = movie_id_to_index[movie_id]

    candidates = []

    for movie_idx, score in zip(
        cbf_indices[idx],
        cbf_scores[idx]
    ):

        candidates.append(
            (
                index_to_movie_id[int(movie_idx)],
                float(score)
            )
        )

    return candidates[:top_n]

In [29]:
def get_cf_candidates(movie_id, top_n=50):

    if movie_id not in movie_id_to_index:
        return []

    idx = movie_id_to_index[movie_id]

    candidates = []

    for movie_idx, score in zip(
        cf_indices[idx],
        cf_scores[idx]
    ):

        candidates.append(
            (
                index_to_movie_id[int(movie_idx)],
                float(score)
            )
        )

    return candidates[:top_n]

In [30]:
def hybrid_recommend(
    user_id,
    seed_movie_id,
    top_n=5
):

    # Default alpha
    alpha = ALPHA

    # User cold start
    user_count = user_rating_counts.get(
        user_id,
        0
    )

    if user_count < 3:
        alpha = 1.0

    # Item cold start
    seed_rating_count = rating_counts.get(
        seed_movie_id,
        0
    )

    if seed_rating_count < 5:
        alpha = 1.0

    cbf = dict(
        get_cbf_candidates(
            seed_movie_id,
            TOP_N
        )
    )

    cf = dict(
        get_cf_candidates(
            seed_movie_id,
            TOP_N
        )
    )

    candidate_movies = (
        set(cbf.keys()) |
        set(cf.keys())
    )

    results = []

    for movie_id in candidate_movies:

        cbf_score = cbf.get(
            movie_id,
            0
        )

        cf_score = cf.get(
            movie_id,
            0
        )

        final_score = (
            alpha * cbf_score +
            (1 - alpha) * cf_score
        )

        results.append(
            (
                movie_id,
                final_score,
                cbf_score,
                cf_score
            )
        )

    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return results[:top_n]

In [31]:
seed_movie = movies.iloc[0]

print(seed_movie)

movieId                                              1
title                                 Toy Story (1995)
genres     Adventure|Animation|Children|Comedy|Fantasy
Name: 0, dtype: object


In [32]:
test_user = ratings.iloc[0]["userId"]

print("User:", test_user)

User: 1.0


In [33]:
recommendations = hybrid_recommend(
    user_id=int(test_user),
    seed_movie_id=int(seed_movie["movieId"]),
    top_n=5
)

recommendations

[(2355, 0.27476614713668823, 0.5495322942733765, 0),
 (45074, 0.20965495705604553, 0.41930991411209106, 0),
 (4886, 0.20965495705604553, 0.41930991411209106, 0),
 (65577, 0.20965495705604553, 0.41930991411209106, 0),
 (166461, 0.20965495705604553, 0.41930991411209106, 0)]

In [34]:
recommendation_df = pd.DataFrame(
    recommendations,
    columns=[
        "movieId",
        "final_score",
        "cbf_score",
        "cf_score"
    ]
)

recommendation_df = recommendation_df.merge(
    movies[
        ["movieId", "title", "genres"]
    ],
    on="movieId",
    how="left"
)

recommendation_df

,movieId,final_score,cbf_score,cf_score,title,genres
0,2355,0.274766,0.549532,0,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy
1,45074,0.209655,0.419310,0,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
2,4886,0.209655,0.419310,0,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
3,65577,0.209655,0.419310,0,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
4,166461,0.209655,0.419310,0,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy


In [35]:
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

print("Train:", train_ratings.shape)
print("Test:", test_ratings.shape)

Train: (80668, 4)
Test: (20168, 4)


In [36]:
user_mean_train = (
    train_ratings
    .groupby("userId")["rating"]
    .mean()
)

global_mean = train_ratings["rating"].mean()

predictions = []

actuals = []

for row in test_ratings.itertuples():

    actual = row.rating

    predicted = user_mean_train.get(
        row.userId,
        global_mean
    )

    predictions.append(predicted)
    actuals.append(actual)

In [37]:
mae = mean_absolute_error(
    actuals,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        actuals,
        predictions
    )
)

print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))

MAE : 0.7386
RMSE: 0.9497


In [38]:
K = 5
RELEVANT_THRESHOLD = 4.0

In [39]:
def get_user_relevant_movies(
    user_id,
    rating_data
):

    user_data = rating_data[
        rating_data["userId"] == user_id
    ]

    return set(
        user_data[
            user_data["rating"] >=
            RELEVANT_THRESHOLD
        ]["movieId"]
    )

In [40]:
precisions = []
recalls = []

users = test_ratings["userId"].unique()

for user_id in users[:100]:

    user_train = train_ratings[
        train_ratings["userId"] == user_id
    ]

    if len(user_train) == 0:
        continue

    seed_movie_id = int(
        user_train.iloc[-1]["movieId"]
    )

    recommendations = hybrid_recommend(
        user_id=int(user_id),
        seed_movie_id=seed_movie_id,
        top_n=K
    )

    recommended_movies = set(
        x[0]
        for x in recommendations
    )

    relevant_movies = get_user_relevant_movies(
        user_id,
        test_ratings
    )

    if len(relevant_movies) == 0:
        continue

    hits = (
        recommended_movies &
        relevant_movies
    )

    precision = (
        len(hits) / K
    )

    recall = (
        len(hits) /
        len(relevant_movies)
    )

    precisions.append(precision)
    recalls.append(recall)

In [41]:
precision_at_5 = np.mean(precisions)
recall_at_5 = np.mean(recalls)

print(
    "Precision@5:",
    round(precision_at_5, 4)
)

print(
    "Recall@5:",
    round(recall_at_5, 4)
)

Precision@5: 0.006
Recall@5: 0.0007


In [42]:
import os

os.makedirs(
    "hybrid_recommender_artifacts",
    exist_ok=True
)

In [43]:
np.save(
    "hybrid_recommender_artifacts/cbf_indices.npy",
    cbf_indices
)

np.save(
    "hybrid_recommender_artifacts/cbf_scores.npy",
    cbf_scores
)

In [44]:
np.save(
    "hybrid_recommender_artifacts/cf_indices.npy",
    cf_indices
)

np.save(
    "hybrid_recommender_artifacts/cf_scores.npy",
    cf_scores
)

In [45]:
joblib.dump(
    tfidf,
    "hybrid_recommender_artifacts/tfidf_vectorizer.pkl"
)

['hybrid_recommender_artifacts/tfidf_vectorizer.pkl']

In [46]:
joblib.dump(
    movie_id_to_index,
    "hybrid_recommender_artifacts/movie_id_to_index.pkl"
)

joblib.dump(
    index_to_movie_id,
    "hybrid_recommender_artifacts/index_to_movie_id.pkl"
)

joblib.dump(
    rating_counts,
    "hybrid_recommender_artifacts/rating_counts.pkl"
)

joblib.dump(
    user_rating_counts,
    "hybrid_recommender_artifacts/user_rating_counts.pkl"
)

['hybrid_recommender_artifacts/user_rating_counts.pkl']

In [47]:
movies.to_csv(
    "hybrid_recommender_artifacts/movies.csv",
    index=False
)

In [48]:
metrics = {
    "MAE": float(mae),
    "RMSE": float(rmse),
    "Precision@5": float(precision_at_5),
    "Recall@5": float(recall_at_5),
    "alpha": ALPHA,
    "top_n": TOP_N,
    "cold_start_user_threshold": 3,
    "cold_start_item_threshold": 5
}

with open(
    "hybrid_recommender_artifacts/metrics.json",
    "w"
) as f:

    json.dump(
        metrics,
        f,
        indent=4
    )

metrics

{'MAE': 0.7386140470646892,
 'RMSE': 0.9497467465720074,
 'Precision@5': 0.006000000000000001,
 'Recall@5': 0.0007344882121446172,
 'alpha': 0.5,
 'top_n': 50,
 'cold_start_user_threshold': 3,
 'cold_start_item_threshold': 5}

In [49]:
config = {
    "alpha": 0.5,
    "top_n": 50,
    "recommendation_k": 5,
    "user_cold_start_threshold": 3,
    "item_cold_start_threshold": 5,
    "relevant_rating_threshold": 4.0,
    "method": "Hybrid CBF + CF",
    "similarity": "Pearson Correlation"
}

with open(
    "hybrid_recommender_artifacts/config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )